In [0]:
env="p"

In [0]:
spark.sql(f"""
    UPDATE `{env}-cdh`.primaryactive.interaction
    SET    PortfolioKey  = 3031,
           PortfolioName = 'DOMUS PORTFOLIO - D'
    WHERE  PortfolioKey IS NULL
    AND    substring(cdi, 1, 3) = 'DMD'
""")

In [0]:
spark.sql(f"""
    UPDATE `{env}-cdh`.primaryactive.interaction
    SET    PortfolioKey  = 2829,
           PortfolioName = 'RHODIUM PORTFOLIO - D'
    WHERE  PortfolioKey IS NULL
    AND    substring(cdi, 1, 3) = 'RDD'
""")

## ITL-117

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativeactive.dpaymentschedule
    ADD COLUMN PartyKey STRING after ExposureKey
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativeactive.dpaymentschedule
    ADD COLUMN AgreementKey STRING after PartyKey
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativeactive.dpaymentschedule
    ADD COLUMN PayTolerance int after SettlementCaseRecordKey
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativeactive.dpaymentschedule
    ADD COLUMN PayPercent decimal(12,6) after PayTolerance
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativeactive.dpaymentschedule
    ADD COLUMN PayPercentAmount decimal(18,2) after PayPercent
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativearchive.dpaymentschedule
    ADD COLUMN PartyKey STRING after ExposureKey
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativearchive.dpaymentschedule
    ADD COLUMN AgreementKey STRING after PartyKey
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativearchive.dpaymentschedule
    ADD COLUMN PayTolerance int after SettlementCaseRecordKey
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativearchive.dpaymentschedule
    ADD COLUMN PayPercent decimal(12,6) after PayTolerance
""")

In [0]:
spark.sql(f"""
    ALTER TABLE `{env}-cdh`.derivativearchive.dpaymentschedule
    ADD COLUMN PayPercentAmount decimal(18,2) after PayPercent
""")

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

